In [ ]:
# TreeShrew renalaysis

In [ ]:
# All cells UMAP and clustering analysis

In [ ]:
import snapatac2 as snap
import numpy as np
import pandas as pd
import scanpy as sc
import polars as pl
import os 
snap.__version__

In [ ]:
os.chdir('AllCells_UMAP_snapATAC2')

In [ ]:
myCustomGenome = snap.genome.Genome(fasta='/home/sunkeyong/softeware/ref_tree_shrew/TreeShrew_scATAC_ref/fasta/genome.fa', 
                                    annotation='/home/sunkeyong/softeware/ref_tree_shrew/TreeShrew_scATAC_ref/genes/genes.gtf')

In [ ]:
# 创建一个样本数据框以路径和样本名称初始化
sample= ["Sample.list"]
path = ["pathway/Sample.list.bed.gz"]

In [ ]:
%%time
data = snap.pp.import_data(
    path,
    file=[name + '.h5ad' for name in sample],
    chrom_sizes=myCustomGenome,
    sorted_by_barcode=True,
    min_num_fragments=5,
    whitelist="./TS_Keep_Barcodes.csv",
    n_jobs=3,
)
data

In [ ]:
snap.pp.add_tile_matrix(data, bin_size=500, n_jobs=3)

In [ ]:
%%time
#[name + '.h5ad' for name in sample]
xdata = snap.AnnDataSet(
    adatas=[(name, adata) for name, adata in zip(sample, data)],
    filename='TreeShrew.h5ads'
)
xdata

In [ ]:
print(f'Number of cells: {xdata.n_obs}')
print(f'Number of unique barcodes: {np.unique(xdata.obs_names).size}')

In [ ]:
snap.pp.select_features(xdata, n_features=250000, n_jobs=8)
snap.tl.spectral(xdata)
snap.tl.umap(xdata)

In [ ]:
snap.pl.umap(xdata, color="sample", interactive=False)

In [ ]:
snap.pp.knn(xdata, use_rep="X_spectral")

In [ ]:
snap.tl.leiden(xdata,resolution=1,key_added='leiden_R1')
snap.tl.leiden(xdata,resolution=2,key_added='leiden_R2')
snap.tl.leiden(xdata,resolution=3,key_added='leiden_R3')

In [ ]:
snap.pl.umap(xdata, color="leiden_R1", interactive=False)

In [ ]:
df2 = pd.DataFrame(xdata.obsm["X_umap"],columns = ["x","y"])
df2
df = pd.DataFrame({'sample':xdata.obs["sample"],
                   'leiden_R1':xdata.obs["leiden_R1"],
                   'leiden_R2':xdata.obs["leiden_R2"],
                   'leiden_R3':xdata.obs["leiden_R3"],
                   'cell':xdata.obs_names},)
df
dfdf = pd.concat([df,df2],axis=1)
dfdf.to_csv("snap_metadata.csv")

In [ ]:
xdata.close()
xdata